In [14]:
import openai, json, requests

client = openai.OpenAI()
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [15]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
    "get_movie_credits": get_movie_credits,
}

In [16]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

SYSTEM_PROMPT = """You are a personalized movie recommendation chatbot.
You remember the user's favorite genres and movies they have already watched.
Never recommend movies the user has already seen.
Only recommend movies when the user explicitly asks for a recommendation.
Always respond in the same language the user used in their message.
You have access to the following functions to get up-to-date movie information:
- get_popular_movies(): Returns a list of currently popular movies.
- get_movie_details(id): Returns detailed information about a specific movie by its ID.
- get_movie_credits(id): Returns the cast and crew of a specific movie by its ID.
- get_similar_movies(id): Returns a list of movies similar to a given movie by its ID.
Use these functions when you need movie data to answer questions or make better recommendations."""

In [17]:
def process_ai_response(message, messages):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Agent: [{function_name} with args: {arguments} 호출]")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        call_ai(messages)
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai(messages):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message, messages)


def chat(user_input):
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai(messages)

In [18]:
# 대화 시작 (messages 초기화 — 리셋하려면 이 셀부터 다시 실행)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

# Turn 1: 인기 영화 목록 요청
chat("지금 인기 있는 영화 알려줘")

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies with args: {} 호출]
Agent: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Shelter**
   - 개요: 한 남자가 자진 망명 중인 외딴 섬에서 폭풍우에 휘말린 한 소녀를 구하면서 숨겨진 과거의 적들과 맞서 싸워야 하는 이야기를 다룹니다.
   - 개봉일: 2026-01-28
   - 평점: 6.96
   - ![포스터](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **Mercy**
   - 개요: 근미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있으며, 진실을 증명할 시간이 90분밖에 없습니다.
   - 개봉일: 2026-01-20
   - 평점: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

3. **The Orphans**
   - 개요: 어린 시절 친구였던 두 남자가 사랑의 죽음으로 인해 재회하여 복수를 위해 힘을 합치는 이야기입니다.
   - 개봉일: 2025-08-20
   - 평점: 6.18
   - ![포스터](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개요: 한 섬에서 평화롭게 살던 여자가 복수심에 불타는 옛 선장과 마주치면서 벌어지는 이야기를 다룹니다.
   - 개봉일: 2026-02-17
   - 평점: 5.83
   - ![포스터](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개요: 가족과 휴가 중에 지역 남자들에 의해 공격받고 자매를 잃은 여자가 복수를 결심하는 이야기입니다.
   - 개봉일: 2025-06

In [19]:
# Turn 2: 특정 영화 상세 정보 요청
chat("Mercy에 대해 더 알려줘")

User: Mercy에 대해 더 알려줘
Agent: [get_movie_details with args: {"id": 1236153} 호출]
Agent: [get_movie_credits with args: {"id": 1236153} 호출]
Agent: **영화: Mercy**

- **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있습니다. 그는 자신의 무죄를 증명하기 위해 그가 지지했던 고급 인공지능 판사에게 90분의 시간이 주어지며, 그 판사가 그의 운명을 결정하게 됩니다.
  
- **개봉일**: 2026년 1월 20일  
- **러닝타임**: 99분  
- **장르**: 사이언스 픽션, 액션, 스릴러  
- **예산**: 60,000,000 USD  
- **수익**: 49,463,290 USD  
- **평점**: 7.13 (투표 수: 538)

- **태그라인**: "Prove your innocence to an AI judge or face execution." (AI 판사에게 무죄를 증명하거나 사형을 맞이하세요.)

- **링크**: [Mercy - Amazon](https://www.amazon.com/salp/mercy?hhf=)

- **포스터**:
  ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

### 출연진
- **Chris Pratt** - Chris Raven  
  ![Chris Pratt](https://image.tmdb.org/t/p/w185/cRH6HPAQ98PlOwwEvhYO4CM9lwu.jpg)
  
- **Rebecca Ferguson** - Judge Maddox  
  ![Rebecca Ferguson](https://image.tmdb.org/t/p/w185/lJloTOheuQSirSLXNA3JHsrMNfH.jpg)

- **Kali Reis** - Jacqueline 'Jaq' Diallo  
  !

In [20]:
# Turn 3: 비슷한 영화 추천 요청 w/메모리
chat("비슷한 영화 추천해 줄래?")

User: 비슷한 영화 추천해 줄래?
Agent: [get_similar_movies with args: {"id":1236153} 호출]
Agent: 다음은 영화 **Mercy**와 비슷한 영화 추천 목록입니다:

1. **Virtuosity** (1995)
   - 개요: 법 집행 기술 진보 센터에서 개발된 SID라는 가상 현실 실체가 150명의 연쇄 살인자의 성격으로 구성되어 있으며, 단 한 명의 남자만이 그를 저지할 수 있습니다.
   - 평점: 5.50
   - ![포스터](https://image.tmdb.org/t/p/w780/5emgnXYrXjqyiup2JsvtuENseiV.jpg)

2. **Thesis** (1996)
   - 개요: 영상 폭력에 대한 학위를 작성하던 학생이 한 소녀가 고문당하다 죽는 스너프 비디오를 발견하게 되면서 벌어지는 이야기입니다.
   - 평점: 7.47
   - ![포스터](https://image.tmdb.org/t/p/w780/dPyONW7fyb3iQ5s4WieNaHSX2id.jpg)

3. **Trapped** (2002)
   - 개요: 딸이 납치된 제닝스 가족이 범죄자들을 상대로 역공격을 펼치는 이야기입니다.
   - 평점: 6.16
   - ![포스터](https://image.tmdb.org/t/p/w780/4sujD0KP5O90TXAWrTjqDPBTnfQ.jpg)

4. **S.W.A.T.** (2003)
   - 개요: 로스앤젤레스 경찰의 엘리트 부대가 범죄 보스의 호위 임무를 수행하면서 벌어지는 사건을 다룹니다.
   - 평점: 6.15
   - ![포스터](https://image.tmdb.org/t/p/w780/bon63yPVIgUFLP2653Dg9GCOJLJ.jpg)

5. **Hancock** (2008)
   - 개요: 하루가 멀다 하고 큰 피해를 입히며 구원 활동을 하는 슈퍼히어로가 그의 이미지를 개선하기 위해 PR 전문가를 고용하게 되는 이야기입니다.
   - 평점: 6.35
   - 

In [22]:
# Turn 4: 메모리 확인
chat("지금까지 어떤 영화들 얘기했지?")

User: 지금까지 어떤 영화들 얘기했지?
Agent: 지금까지 언급된 영화 목록은 다음과 같습니다:

### 인기 영화 목록
1. **Shelter**
2. **Mercy**
3. **The Orphans**
4. **The Bluff**
5. **A Woman Scorned**

### 영화 **Mercy**에 대한 정보
- **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있으며, AI 판사에게 무죄를 증명해야 하는 이야기.
- **출연진**: Chris Pratt, Rebecca Ferguson, Kali Reis, Kylie Rogers, Annabelle Wallis 등.

### 비슷한 영화 추천 목록
1. **Virtuosity**
2. **Thesis**
3. **Trapped**
4. **S.W.A.T.**
5. **Hancock**

이 외에 더 궁금한 영화나 다른 질문이 있으시면 언제든지 말씀해 주세요!
